In [1]:
import cvxpy as cp
import numpy as np

# --- Helper Functions ---
def partial_trace_one_qubit(rho, trace_qubit, num_qubits):
    full_dim = 2 ** num_qubits
    reduced_dim = 2 ** (num_qubits - 1)
    reduced_rho = [[0 for _ in range(reduced_dim)] for _ in range(reduced_dim)]
    for i in range(full_dim):
        for j in range(full_dim):
            bi = [(i >> k) & 1 for k in reversed(range(num_qubits))]
            bj = [(j >> k) & 1 for k in reversed(range(num_qubits))]
            if bi[trace_qubit] == bj[trace_qubit]:
                bi_reduced = bi[:trace_qubit] + bi[trace_qubit+1:]
                bj_reduced = bj[:trace_qubit] + bj[trace_qubit+1:]
                row = sum([bit << (num_qubits - 2 - k) for k, bit in enumerate(bi_reduced)])
                col = sum([bit << (num_qubits - 2 - k) for k, bit in enumerate(bj_reduced)])
                reduced_rho[row][col] += rho[i, j]
    return cp.vstack([cp.hstack(row) for row in reduced_rho])

def partial_transpose_qubits(rho, transpose_qubits, num_qubits):
    dim = 2 ** num_qubits
    rho_pt = [[0 for _ in range(dim)] for _ in range(dim)]
    for i in range(dim):
        for j in range(dim):
            bi = [(i >> k) & 1 for k in reversed(range(num_qubits))]
            bj = [(j >> k) & 1 for k in reversed(range(num_qubits))]
            bi_new, bj_new = bi.copy(), bj.copy()
            for q in transpose_qubits:
                bi_new[q], bj_new[q] = bj[q], bi[q]
            row_new = sum([bit << (num_qubits - 1 - k) for k, bit in enumerate(bi_new)])
            col_new = sum([bit << (num_qubits - 1 - k) for k, bit in enumerate(bj_new)])
            rho_pt[row_new][col_new] += rho[i, j]
    return cp.vstack([cp.hstack(row) for row in rho_pt])

# --- Fixed Parameters ---
epsilon = 0
K = 10
identity_16 = np.eye(16)
identity_2 = np.eye(2)
mathematica_data = []

# --- Loop over q values ---
for q in np.arange(0.50, 1.01, 0.01):
    print(f"\n--- Solving for q = {q:.2f} ---")
    
    # --- Define states for this q ---
    rho_12 = np.array([
        [(1+2*q)/6, 0, 0, (-1+4*q)/6],
        [0, (1-q)/3, 0, 0],
        [0, 0, (1-q)/3, 0],
        [(-1+4*q)/6, 0, 0, (1+2*q)/6]
    ])
    
    sigma_34 = np.array([
        [(1+2*q)/6, 0, 0, (-1+4*q)/6],
        [0, (1-q)/3, 0, 0],
        [0, 0, (1-q)/3, 0],
        [(-1+4*q)/6, 0, 0, (1+2*q)/6]
    ])
    
    rho_sigma = np.kron(rho_12, sigma_34)

    # --- Variables ---
    M = [cp.Variable((16, 16), hermitian=True) for _ in range(K)]

    # --- Compute p value ---
    sum_Tr4_Mi_tensor_I2 = 0
    for M_i in M:
        Tr4_Mi = partial_trace_one_qubit(M_i, trace_qubit=3, num_qubits=4)
        Tr4_Mi_tensor_I2 = cp.kron(Tr4_Mi, identity_2)
        sum_Tr4_Mi_tensor_I2 += Tr4_Mi_tensor_I2
    p_expr = 2 * cp.real(cp.trace(sum_Tr4_Mi_tensor_I2 @ rho_sigma))

    # --- Objective ---
    objective_terms = sum(cp.real(cp.trace(M_i @ rho_sigma)) for M_i in M)
    objective = cp.Maximize((1 - p_expr)/2 + objective_terms)

    # --- Constraints ---
    constraints = [2 * cp.real(sum_Tr4_Mi_tensor_I2) <= identity_16]
    for M_i in M:
        constraints += [
            M_i >> 0,
            M_i << (1 - epsilon) * identity_16,
            partial_transpose_qubits(M_i, [1, 2], 4) >> 0
        ]

    # --- Solve ---
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.MOSEK, verbose=False)

    print("Optimal value:", problem.value)
    print("p value:", p_expr.value)
    print("Objective term:", objective_terms.value)

    if problem.status == cp.OPTIMAL:
        mathematica_data.append([round(q, 2), round(problem.value, 6)])
    else:
        mathematica_data.append([round(q, 2), None])

print("\nMathematica-ready output:\n{")
for i, pair in enumerate(mathematica_data):
    sep = "," if i < len(mathematica_data) - 1 else ""
    val_str = "Null" if pair[1] is None else f"{pair[1]:.6f}"
    print(f"  {{{pair[0]:.2f}, {val_str}}}{sep}")
print("}")




--- Solving for q = 0.50 ---


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\cvxpy\problems\problem.py:164: UserWarning: Constraint #3 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn(f"Constraint #{i} contains too many subexpressions. "
c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\cvxpy\problems\problem.py:164: UserWarning: Constraint #6 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn(f"Constraint #{i} contains too many subexpressions. "
c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\cvxpy\problems\problem.py:164: UserWarning: Constraint #9 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn(f"Constraint #{i} contains too many subexpressions. "
c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\cvxpy\problems\problem.py:164: UserWarning: Cons

Optimal value: 0.49999999999998784
p value: 6.764333209489763e-14
Objective term: 2.182912579193878e-14

--- Solving for q = 0.51 ---
Optimal value: 0.49999999999997496
p value: 1.4223386377593228e-13
Objective term: 4.634625400170023e-14

--- Solving for q = 0.52 ---
Optimal value: 0.499999999999956
p value: 2.5788428440967975e-13
Objective term: 8.51268953347288e-14

--- Solving for q = 0.53 ---
Optimal value: 0.4999999999999327
p value: 4.09857537348661e-13
Objective term: 1.3760545551379917e-13

--- Solving for q = 0.54 ---
Optimal value: 0.49999999999985806
p value: 8.956388875790945e-13
Objective term: 3.05680422445076e-13

--- Solving for q = 0.55 ---
Optimal value: 0.49999999999972344
p value: 1.7998903835190355e-12
Objective term: 6.233846171393633e-13

--- Solving for q = 0.56 ---
Optimal value: 0.4999999999991841
p value: 5.4065813265435874e-12
Objective term: 1.8873940834733902e-12

--- Solving for q = 0.57 ---
Optimal value: 0.49999999999851663
p value: 1.0038884203839453e